*Every agent that ever played this competition is still in the record, dated, with its actions intact. That makes a question tractable here that is hard everywhere else: not who is winning, but who is descended from whom, and what happened to the ones that are gone.*

**What this is about.** [An earlier notebook](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay) measured how much a single agent varies from its own past. This one measures how the lineages relate to each other, over 32,570 episodes and the whole life of the ladder so far.

**The result in one line.** The field does not drift. **It turns over by sweeps.** A lineage appears, takes a quarter to a half of every seat played within two days, holds for three, and is gone. Three complete sweeps happened in nine days.

**What that costs everyone.** Any instrument calibrated against "the field" goes stale in days rather than weeks, and section 4 measures it against a control: a pool of opponents recorded on one day covers about half of the seats played two days later and under a twentieth of the seats played a week later. A pool, a win rate, a corpus of recorded routes: all of them are measuring a population that will not exist when you use the answer.

**And one methodological finding that gates the rest**, in section 1. The obvious way to identify a lineage is by hashing its action stream. Measured, the full-length hash changes **every single game** for the median agent: it identifies the episode, not the agent. Anything built on it would give every game its own species.

---

## What was already known

Three earlier results set this up, and each came from treating the ladder as a population rather than as a set of independent optimisers.

**There are species.** [The leaderboard behaves like a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient): what you meet at one rating band is not what you meet at another.

**Much of the field shares an opening.** [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) finds unrelated teams emitting identical actions, turn for turn, deep into the episode. Common descent, not convergence.

**A route splits into a plan and a market channel.** [The genetic potential of a replay](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay) shows an agent can vary its trading on 140 turns while building the identical farm. That one matters immediately here: it is why the naive lineage marker fails.

In [ ]:
import json, sys, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

def find_dir(marker, roots=("/kaggle/input", ".", "..", "data")):
    hits = []
    for r in roots:
        p = Path(r)
        if p.exists():
            hits += [q.parent for q in p.rglob(marker)]
    if not hits:
        raise FileNotFoundError(f"could not find {marker!r} under {roots}")
    return sorted(hits, key=lambda q: len(str(q)))[0]

FIELD = find_dir("stream_hashes.csv")     # georgymamarin/kaggriculture-episodes
TOP = find_dir("streams.parquet")         # destbreso/kaggriculture-top10-replay-streams
print("field  :", FIELD)
print("top ten:", TOP)

hashes = pd.read_csv(FIELD / "stream_hashes.csv",
                     dtype={"episode_id": str, "seat": str})
eps = pd.read_csv(FIELD / "episodes.csv",
                  dtype={"episode_id": str, "sub_0": str, "sub_1": str,
                         "team_0": str, "team_1": str})
eps = eps[(eps.type == "EPISODE_TYPE_PUBLIC") & (eps.state == "COMPLETED")].copy()
eps["day"] = eps.create_time.str[:10]
print(f"\n{len(eps):,} public completed episodes, "
      f"{len(hashes):,} hashed seats, {eps.day.min()} to {eps.day.max()}")

## 1. Which marker identifies a lineage? Not the obvious one

The dataset hashes each seat's action stream at five prefix lengths: turns 24, 100, 200, 400 and the full 719. The obvious choice for identifying an agent is the longest one, since it uses the most information.

It is the worst one available, and the reason is the earlier result. An agent whose construction plan never moves still varies its market orders on a hundred turns or more, so its full-length hash changes every game while the agent has not changed at all.

**The test is simple: take agents with at least ten recorded seats and count how many distinct hashes each one has.** A marker of identity is stable within an agent. A marker of noise is not.

In [ ]:
h = hashes.set_index(["episode_id", "seat"])
PREF = ["stream_h24", "stream_h100", "stream_h200", "stream_h400", "stream_h719"]

per_sub = defaultdict(list)
for seat in ("0", "1"):
    m = eps[["episode_id", f"sub_{seat}", "day"]].copy()
    m["seat"] = seat
    m = m.merge(hashes, on=["episode_id", "seat"], how="inner")
    for sub, g in m.groupby(f"sub_{seat}"):
        per_sub[sub].append(g)

groups = {s: pd.concat(v) for s, v in per_sub.items()}
big = {s: g for s, g in groups.items() if len(g) >= 10}
print(f"{len(groups):,} submissions with hashed seats, "
      f"{len(big):,} with at least ten\n")

rows = []
for p in PREF:
    frac, ones = [], 0
    for s, g in big.items():
        d = g[p].nunique(dropna=True)
        if d:
            frac.append(d / len(g))
            ones += d == 1
    rows.append({"prefix": p.replace("stream_", ""),
                 "median distinct per seat": round(float(np.median(frac)), 3),
                 "agents with exactly one": f"{100*ones/len(big):.1f} %"})
marker = pd.DataFrame(rows)
display(marker.style.hide(axis="index"))

In [ ]:
# Lightness separated and monotone, so every figure below survives colour vision
# deficiency, greyscale printing and a bad projector.
LINE = ["#B45309", "#1D4ED8", "#1B7F5F", "#7C2D12", "#6D28D9", "#0E7490"]
GOOD, BAD, INK, PALE = "#1B7F5F", "#0F172A", "#334155", "#E7E4DE"

fig, ax = plt.subplots(figsize=(10, 4.6))
x = np.arange(len(PREF))
y = [float(v) for v in marker["median distinct per seat"]]
cols = [GOOD if v < 0.2 else ("#B45309" if v < 0.8 else BAD) for v in y]
ax.bar(x, y, color=cols, width=0.62)
ax.axhline(1.0, color=BAD, lw=1.2, ls="--")
ax.text(-0.44, 1.04, "a new hash every single game: identifies the EPISODE",
        fontsize=9.5, color=BAD)
ax.axhline(0.1, color=GOOD, lw=1.2, ls="--")
ax.text(-0.44, 0.13, "one hash per ten games: identifies the AGENT",
        fontsize=9.5, color=GOOD)
for i, v in enumerate(y):
    ax.text(i, v + 0.04, f"{v:.3f}", ha="center", fontsize=10, color=INK)
ax.set_xticks(x)
ax.set_xticklabels([p.replace("stream_h", "hashed to turn ") for p in PREF],
                   fontsize=9.5)
ax.set_ylabel("distinct hashes per recorded seat, median", fontsize=9.5)
ax.set_ylim(0, 1.22)
ax.set_title("A longer hash is not a better marker. It is a worse one.",
             loc="left", fontsize=13, pad=10)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
plt.show()

**The full-length hash has a median of exactly 1.000 distinct values per seat.** One new hash every game. It is a perfect identifier of the episode and a useless identifier of the agent.

The 24-turn prefix is stable for four agents in five. So the openings are the lineage markers, and everything below uses turns 24, 100 and 200 and refuses the last two. That is not a preference; it is the difference between a phylogeny and a list of games.

## 2. Does sharing an opening mean anything? A statistic that reads backwards

The census result usually quoted is that **94 % of seats share a schedule at turn 24**. Read quickly, that says the opening is forced and agreement is worthless as evidence.

It says nothing of the kind, and the difference matters enough to compute.

In [ ]:
allseats = []
for seat in ("0", "1"):
    m = eps[["episode_id"]].copy()
    m["seat"] = seat
    allseats.append(m)
allseats = pd.concat(allseats).merge(hashes, on=["episode_id", "seat"], how="inner")

rows = []
for p in ("stream_h24", "stream_h400"):
    v = allseats[p].dropna()
    c = v.value_counts()
    pairs = len(v) * (len(v) - 1) / 2
    same = (c * (c - 1) / 2).sum()
    rows.append({
        "prefix": p.replace("stream_", ""),
        "seats": f"{len(v):,}",
        "distinct schedules": f"{len(c):,}",
        "seats sharing with at least one other": f"{100*c[c>=2].sum()/len(v):.1f} %",
        "the single most common schedule": f"{100*c.iloc[0]/len(v):.1f} %",
        "PAIRS of seats that agree": f"{100*same/pairs:.1f} %"})
display(pd.DataFrame(rows).set_index("prefix").T)

**"94 % share a schedule" means 94 % are not unique.** It does not mean 94 % play the same opening. There are thousands of distinct openings at turn 24, the most popular one covers about a seventh of the field, and **only one pair of seats in eighteen actually agrees**. By turn 400 it is one pair in a thousand.

So an opening match is real evidence after all. The care that survives is different and weaker: openings are the part of a route most easily adopted wholesale from a public notebook, so a large group sharing one is a clade of adoption rather than a family of gradual descent. **Build from where lineages stop agreeing, and weight a deep split far above a shallow one.**

## 3. The control: how far does an agent diverge from itself?

Before any split between two agents can mean anything, there has to be a floor. Two runs of one agent already differ, because weeds spawn in different places and harvests land on different turns.

**So: at what turn do two episodes of the SAME agent first disagree?** Anything shallower than that, between two different agents, is inside the noise a single lineage generates on its own.

This uses the ten highest rated submissions, at eight of their own episodes each, on the plan channel only.

In [ ]:
top = pd.read_parquet(TOP / "streams.parquet")
top["episode_id"] = top["episode_id"].astype(str)
top["submission"] = top["submission"].astype(str)
top["plan"] = top["farmer"] + "|" + top["hands_h"]
tmeta = pd.read_csv(TOP / "episodes.csv",
                    dtype={"episode_id": str, "submission": str})

# key on the PAIR: an episode between two of the ten appears once per seat
by_ep = {k: g.sort_values("turn").plan.tolist()
         for k, g in top.groupby(["submission", "episode_id"])}

floors = []
for sub, g in tmeta.groupby("submission"):
    seat = g.seat.value_counts().idxmax()
    ids = g[g.seat == seat].sort_values("episode_id").episode_id.head(8).tolist()
    runs = [by_ep[(sub, e)] for e in ids if (sub, e) in by_ep]
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            a, b = runs[i], runs[j]
            n = min(len(a), len(b))
            d = next((t for t in range(n) if a[t] != b[t]), n)
            floors.append(d)
floors = np.array(floors)
print(f"{len(floors)} pairs of episodes from the SAME agent\n")
for q in (0, 10, 50, 90, 100):
    print(f"  p{q:<4} first disagreement at turn {int(np.percentile(floors, q)):>4}")
FLOOR = int(np.percentile(floors, 50))
print(f"\n  An agent splits from ITSELF at turn {FLOOR}, half the time.")
print("  Any between-agent split shallower than that is not evidence of "
      "separate ancestry.")

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.3))
ax.hist(floors, bins=np.arange(0, 745, 24), color=LINE[2],
        label="two episodes of the SAME agent")
top_y = ax.get_ylim()[1]
ax.axvspan(0, FLOOR, color=BAD, alpha=0.07)
ax.axvline(FLOOR, color=BAD, lw=1.7)
ax.text(FLOOR + 12, top_y * 0.92,
        f"median: an agent splits from itself at turn {FLOOR}",
        fontsize=10.5, color=BAD, va="center")
ax.text(10, top_y * 0.55,
        "a split between two DIFFERENT\nagents anywhere in here is\ninside the "
        "noise one lineage\nmakes on its own",
        fontsize=9.5, color=INK, va="top")
ax.set_xlim(0, 720)
ax.set_xlabel("turn at which the two episodes first disagree", fontsize=9.5)
ax.set_ylabel("pairs of episodes", fontsize=9.5)
ax.set_title("The floor, without which no split depth means anything",
             loc="left", fontsize=13, pad=10)
ax.legend(frameon=False, fontsize=9.5, loc="upper right")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
plt.show()

## 4. The population history, which is the point of the notebook

Place every submission into a lineage by its opening, nested three deep: the turn-24 group, then the turn-100 group inside it, then turn-200. Then count what share of every day's play each lineage held.

A submission is placed by its **modal** hash rather than any single game, so one odd episode cannot move an agent between lineages.

In [ ]:
MIN_SEATS = 3
USABLE = ["stream_h24", "stream_h100", "stream_h200"]

placed, seats_of, born = {}, {}, {}
for sub, g in groups.items():
    if len(g) < MIN_SEATS:
        continue
    key = []
    for p in USABLE:
        c = g[p].dropna()
        if c.empty:
            break
        key.append(c.mode().iloc[0])
    if len(key) == len(USABLE):
        placed[sub] = tuple(key)
        seats_of[sub] = g
        born[sub] = g.day.min()

print(f"{len(placed):,} submissions placed into a nested lineage\n")
for lvl, p in enumerate(USABLE):
    n = len({v[:lvl+1] for v in placed.values()})
    top3 = Counter(v[:lvl+1] for v in placed.values()).most_common(3)
    print(f"  {p.replace('stream_',''):<6} {n:>5} lineages   largest "
          + ", ".join(f"{c} ({100*c/len(placed):.0f} %)" for _, c in top3))

In [ ]:
by_day = defaultdict(Counter)
for sub, g in seats_of.items():
    k = placed[sub][0]
    for d, n in g.day.value_counts().items():
        by_day[d][k] += n
for d in sorted(set(eps.day)):
    tot = int((eps.day == d).sum()) * 2
    if d not in by_day:
        by_day[d] = Counter()

days = sorted(d for d in by_day if sum(by_day[d].values()) >= 400)
size = Counter()
for d in days:
    size.update(by_day[d])
keep = [k for k, _ in size.most_common(5)]

share = {k: [100 * by_day[d][k] / sum(by_day[d].values()) for d in days]
         for k in keep}
other = [100 - sum(share[k][i] for k in keep) for i in range(len(days))]

# order the bands by when each lineage first appears, so the picture reads left
# to right as succession rather than as an arbitrary stack
birth_i = {k: next((i for i, d in enumerate(days) if by_day[d][k] > 0), 0)
           for k in keep}
keep = sorted(keep, key=lambda k: birth_i[k])
share = {k: [100 * by_day[d][k] / sum(by_day[d].values()) for d in days]
         for k in keep}
other = [100 - sum(share[k][i] for k in keep) for i in range(len(days))]

fig, ax = plt.subplots(figsize=(13.5, 6.2))
X = np.arange(len(days))
ax.stackplot(X, other, *[share[k] for k in keep],
             colors=[PALE] + LINE[:len(keep)], edgecolor="white", linewidth=0.8)

# direct labels at each lineage's own peak, so identity never rests on colour
base = np.array(other, dtype=float)
for n, k in enumerate(keep):
    v = np.array(share[k], dtype=float)
    mid = base + v / 2
    i = int(v.argmax())
    if v[i] > 8:
        ax.text(X[i], mid[i], f"{k[:6]}\n{v[i]:.0f}%", ha="center", va="center",
                fontsize=10, color="white", weight="bold", linespacing=1.15)
    base = base + v

ax.set_xticks(X)
ax.set_xticklabels([d[5:] for d in days], fontsize=9)
ax.set_ylim(0, 100)
ax.set_xlim(0, len(days) - 1)
ax.set_ylabel("share of every seat played that day, %", fontsize=10)
ax.set_xlabel("day, 2026", fontsize=10)
ax.set_title("The field does not drift. It turns over by sweeps.",
             loc="left", fontsize=14.5, pad=12)
ax.text(0, 104, "each band is one lineage, ordered by when it first appears; "
                "the pale band is everything else",
        fontsize=10, color=INK)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
plt.show()

print(f"{'day':<12}" + "".join(f"{k[:6]:>10}" for k in keep) + f"{'other':>10}"
      + f"{'seats':>9}")
for i, d in enumerate(days):
    print(f"{d:<12}" + "".join(f"{share[k][i]:>9.0f}%" for k in keep)
          + f"{other[i]:>9.0f}%{sum(by_day[d].values()):>9,}")

Read the bands as lineages and the width as their share of everything played that day.

**Each one rises within about two days, holds for two or three, and collapses.** They do not coexist and they do not settle. The pattern has a name in population genetics, a selective sweep, and the successive version of it is what happens in a population that cannot recombine: a better variant appears, takes over, and is replaced wholesale rather than blended into.

**The practical consequence is severe and it applies to everyone reading this.** A pool of opponents recorded on one day, a win rate measured against it, a corpus of routes collected for comparison: each of those describes a population that is being replaced while the answer is still being computed.

That deserves a number rather than an adjective, and the picture above is not one. So: draw an equal sized sample of seats from each day, so that a thin day is not also a thin pool, and ask what share of a later day's seats belong to a lineage the earlier day had already shown you. **The control is the same measurement inside a single day**, between two disjoint halves of it, which is what this quantity looks like when nothing has changed at all.

In [ ]:
from datetime import date

POOL, REPS = 400, 40                 # seats per sample, and independent draws
rng2 = np.random.default_rng(11)
seats_of_day = {d: np.array([k for k, n in by_day[d].items() for _ in range(n)])
                for d in days}

cov, ctrl = defaultdict(list), []
for _ in range(REPS):
    samp = {d: v[rng2.permutation(len(v))[:POOL]]
            for d, v in seats_of_day.items()}
    for i, a in enumerate(days):
        seen = set(samp[a])
        for b in days[i + 1:]:
            lag = (date.fromisoformat(b) - date.fromisoformat(a)).days
            cov[lag].append(float(np.mean([k in seen for k in samp[b]])))
    for d, v in seats_of_day.items():        # the control, inside one day
        p = rng2.permutation(len(v))
        h = min(POOL, len(v) // 2)
        A, B = set(v[p[:h]]), v[p[len(v) // 2:len(v) // 2 + h]]
        ctrl.append(float(np.mean([k in A for k in B])))

print(f"a pool of {POOL} seats drawn on one day, matched against a later day:")
print(f"  what share of the later day it covers, mean of {REPS} draws\n")
print(f"  same day, two disjoint halves   {np.mean(ctrl):.2f}   <- the control")
lags = [l for l in sorted(cov) if l <= 7 and len(cov[l]) >= 3 * REPS]
ys = [float(np.mean(cov[l])) for l in lags]
for l, y in zip(lags, ys):
    print(f"  {l} day{'s' if l > 1 else ' '} later{'':<19}{y:.2f}"
          f"   ({len(cov[l]) // REPS} pairs of days)")

prev_l, prev_y, half = 0, float(np.mean(ctrl)), None
for l, y in zip(lags, ys):
    if y < 0.5:
        half = prev_l + (prev_y - 0.5) / (prev_y - y) * (l - prev_l)
        break
    prev_l, prev_y = l, y
print(f"\n  it falls through one half at {half:.1f} days")

**Half the field is new within about two days**, and a pool a week old matches under a twentieth of what is being played, against a control of 0.95 for two halves of the same day. The decay is the sweeps seen from the other side: nothing here is drifting slowly enough for a week old sample to describe it.

So a corpus of opponents is not an asset, it is perishable stock, and the shelf life is days. That is not a caveat to add at the end of a study. It is a constraint on what a study can be about.

**Read the number as a property of this record rather than of the ladder.** The dataset is a lagging, uneven sample of what was played, so an unlucky day can put a lineage outside the sample without it having gone anywhere. That biases coverage down, which is why the same day control is printed beside it: 0.95 is how much of the loss is sampling, and the rest is turnover.

### The same thing by the hour, because a day is too coarse to see a sweep arrive

Fourteen daily points hide the shape of the transition. The record is timestamped to the second, so the composition can be evaluated on a rolling window stepped hour by hour, which is smooth enough to watch a lineage take the field and lose it.

The cursor on the upper panel is the moment being shown; the bars below are the instantaneous composition at that moment.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

WINDOW = 18      # hours in the rolling window
STEP = 4         # hours between frames
MINSEATS = 120   # a window thinner than this is noise, not a measurement

ep_hour = eps.set_index("episode_id").create_time.str[:13].to_dict()
hourly = defaultdict(Counter)
for sub, g in seats_of.items():
    k = placed[sub][0]
    for e in g.episode_id:
        h = ep_hour.get(e)
        if h:
            hourly[h][k] += 1
for e, h in ep_hour.items():
    hourly[h]["__all__"] += 2

hours = sorted(hourly)
idx = {h: i for i, h in enumerate(hours)}
K = keep
cnt = np.zeros((len(hours), len(K)))
tot = np.zeros(len(hours))
for h, c in hourly.items():
    i = idx[h]
    tot[i] = c["__all__"]
    for j, k in enumerate(K):
        cnt[i, j] = c[k]

roll_c = np.array([cnt[max(0, i - WINDOW + 1):i + 1].sum(axis=0)
                   for i in range(len(hours))])
roll_t = np.array([tot[max(0, i - WINDOW + 1):i + 1].sum()
                   for i in range(len(hours))])
ok = np.flatnonzero(roll_t >= MINSEATS)
frames = list(range(ok[0], ok[-1] + 1, STEP))
S = np.zeros((len(hours), len(K)))
S[ok] = 100 * roll_c[ok] / roll_t[ok, None]
print(f"{len(frames)} frames, {WINDOW} hour window stepped every {STEP} hours, "
      f"{hours[frames[0]]} to {hours[frames[-1]]}")

fig = plt.figure(figsize=(11, 6.1), dpi=72)
gs = fig.add_gridspec(2, 1, height_ratios=[1.25, 1], hspace=0.42)
axT, axB = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

xs = np.arange(len(hours))
axT.stackplot(xs[ok], *[S[ok, j] for j in range(len(K))],
              colors=LINE[:len(K)], alpha=0.9)
axT.set_ylim(0, 100)
axT.set_xlim(ok[0], ok[-1])
tick = [i for i in ok if hours[i].endswith("T00")]
axT.set_xticks(tick)
axT.set_xticklabels([hours[i][5:10] for i in tick], fontsize=8.5)
axT.set_ylabel("share of seats, %", fontsize=9)
for sp in ("top", "right"):
    axT.spines[sp].set_visible(False)
cursor = axT.axvline(frames[0], color=BAD, lw=1.8)

bars = axB.bar(range(len(K)), [0] * len(K), color=LINE[:len(K)], width=0.62)
axB.set_xticks(range(len(K)))
axB.set_xticklabels([k[:6] for k in K], fontsize=9.5)
axB.set_ylim(0, 65)
axB.set_ylabel("share of seats in the window, %", fontsize=9)
for sp in ("top", "right"):
    axB.spines[sp].set_visible(False)
labels = [axB.text(i, 0, "", ha="center", va="bottom", fontsize=9.5, color=INK)
          for i in range(len(K))]
title = fig.suptitle("", y=0.97, fontsize=13.5)

def draw(i):
    cursor.set_xdata([i, i])
    for j, b in enumerate(bars):
        b.set_height(S[i, j])
        labels[j].set_position((j, S[i, j] + 1.0))
        labels[j].set_text(f"{S[i, j]:.0f}%" if S[i, j] >= 1 else "")
    title.set_text(f"A sweep arriving and leaving, hour by hour"
                   f"      {hours[i].replace('T', '  ')}:00 UTC")
    return [cursor, title, *bars, *labels]

anim = animation.FuncAnimation(fig, draw, frames=frames, interval=110, blit=False)
plt.close(fig)
HTML(anim.to_jshtml(default_mode="loop"))

## 5. Which lineage holds the top, and does the top look different?

Sweeps in the whole field are one thing. What matters competitively is whether the strong submissions belong to one lineage or scatter across many.

In [ ]:
agents = pd.read_csv(FIELD / "agents.csv", dtype={"submission_id": str})
best = agents.groupby("submission_id").rating_after.max()
STRONG = 2900
# only submissions that COULD be attributed to a lineage belong in the
# denominator; counting unplaced ones would make every lineage look small
placed_strong = [s for s in placed if best.get(s, 0) >= STRONG]

rows = []
for k, subs in sorted(Counter(v[0] for v in placed.values()).items(),
                      key=lambda kv: -kv[1])[:8]:
    members = [s for s, v in placed.items() if v[0] == k]
    r = [best.get(s, np.nan) for s in members]
    r = [x for x in r if x == x]
    rows.append({"lineage": k[:6], "submissions": len(members),
                 "first seen": min(born[s] for s in members),
                 "best rating": round(max(r), 1) if r else None,
                 "rated 2900+": sum(1 for x in r if x >= STRONG),
                 "sub-lineages": len({placed[s] for s in members})})
lin = pd.DataFrame(rows)
total_strong = len(placed_strong)
print(f"{int((best >= STRONG).sum())} submissions have ever been rated at or "
      f"above {STRONG}; {total_strong} of them are placed in a lineage\n")
display(lin.style.hide(axis="index"))
print(f"\nthe top lineage holds {lin['rated 2900+'].max()} of the {total_strong} "
      f"placed strong submissions, {100*lin['rated 2900+'].max()/total_strong:.0f} %")
print(f"the LARGEST lineage, {lin.iloc[0]['lineage']} with "
      f"{lin.iloc[0]['submissions']} submissions, holds "
      f"{lin.iloc[0]['rated 2900+']}")

**One lineage holds most of the strong submissions**, and it is not the one that was largest overall. Size and strength are different things here: the biggest sweep by population is not the one that produced the best agents.

That is worth pausing on, because it is the argument against the reflex this competition encourages. Adopting the route everybody is adopting puts you in the largest lineage, which is measurably not the strongest one.

## 6. Is the history a tree? There is barely a history to be a tree

The standard test for treelikeness without building a tree is the delta score of Holland, Huber, Dress and Moulton (2002): for every four lineages there are three ways to pair them up, a perfect tree makes the two largest pairings equal, and delta measures the departure. Zero is a perfect tree, one half is random.

**It cannot be computed from prefix hashes, and that is worth saying rather than working around.** A hash gives a yes or no at each of three depths, so the distance between two lineages takes only four values, every quartet comes out degenerate, and delta returns 0.000 by construction. A number that a method returns whatever the data says is not a measurement.

What the hashes CAN answer is the question delta would have been asked in order to reach: **how much of the field is related to how much of it at all.**

In [ ]:
use = [s for s in placed if len(seats_of[s]) >= 5]
K = [placed[s] for s in use]
n = len(use)
print(f"{n:,} lineages with at least five recorded seats\n")

shared = np.zeros((n, n), dtype=np.int8)
for i in range(n):
    for j in range(n):
        d = 0
        for t in range(len(USABLE)):
            if K[i][t] != K[j][t]:
                break
            d += 1
        shared[i, j] = d
v = shared[np.triu_indices(n, 1)]

print(f"  {'how deep two lineages still agree':<38}{'pairs':>12}{'share':>9}")
for d, name in ((0, "not at all, apart by turn 24"),
                (1, "the turn 24 opening"),
                (2, "through turn 100"),
                (3, "through turn 200")):
    print(f"  {name:<38}{int((v == d).sum()):>12,}{100*(v == d).mean():>8.1f}%")
print(f"\n  for scale, the SAME agent splits from itself at turn {FLOOR}, "
      f"half the time")

In [ ]:
from matplotlib.colors import ListedColormap

# the 150 largest lineages, ordered so that anything related sits together
order = sorted(range(n), key=lambda i: (K[i][0], K[i][1], K[i][2]))
big_i = sorted(order, key=lambda i: -len(seats_of[use[i]]))[:150]
big_i = sorted(big_i, key=lambda i: (K[i][0], K[i][1], K[i][2]))
M = shared[np.ix_(big_i, big_i)]

fig, ax = plt.subplots(figsize=(7.6, 7.2))
cm = ListedColormap([PALE, "#F2C879", "#B45309", "#0F172A"])
ax.imshow(M, cmap=cm, vmin=0, vmax=3, interpolation="nearest")
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel("150 largest lineages, sorted so relatives sit together",
              fontsize=9.5)
ax.set_title("The field, as a matrix. It is almost empty.",
             loc="left", fontsize=13.5, pad=10)
handles = [Patch(facecolor=c, label=l) for c, l in
           ((PALE, "share nothing, apart by turn 24"),
            ("#F2C879", "share the turn 24 opening"),
            ("#B45309", "identical through turn 100"),
            ("#0F172A", "identical through turn 200"))]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=2, frameon=False, fontsize=9)
plt.show()

**Nine pairs of lineages in ten share nothing at all**, differing before turn 24. A thin remainder shares an opening, and a thinner one still is identical for two hundred turns.

That shape is a **star**, not a tree: many unrelated groups radiating from nothing in common, plus a handful of very dense clumps. Gradual divergence, the thing phylogenetics exists to reconstruct, is largely absent here.

**What this field has instead of a history is adoption events.** A route appears, a slice of the field takes it entire, and everything else shares nothing with it or with each other. That is what a population produces when its members can copy one another perfectly and do, and it is why the clumps are so tight and the space between them so empty.

*A companion measurement on real action streams rather than hashes, over 271 lineages, agrees: delta comes out at 0.078 with 74 % of pairs above 0.95 distance. The same star, seen at finer resolution.*

## 7. What is open

**The sweeps have no cause attached.** A lineage falls; nothing here says whether it was beaten head to head, absorbed by its successor, or simply abandoned when its authors moved on. The pairwise result is in the record and can separate those.

**Horizontal transfer is argued and not measured.** Copying is consistent with the tight clusters and so is ordinary common descent. Separating them needs the dates: a lineage carrying a trait its own parent did not have got it sideways.

**An Eve and a LUCA are named but not properly dated.** The lineage holding the strong submissions has a first appearance in the record, but the dataset lags the ladder, so every birth date here is an upper bound. The public notebook that seeded the field is the root candidate and has not been matched to a lineage.

**And the strongest agent on the board belongs to none of it.** The rank 1 submission at the time of the companion notebook's snapshot builds a different farm nearly every time it plays and joins no lineage measured here. Whether that is a descendant that left or something that never belonged is a question about split depth, and it is answerable.

---

**Data.** [`georgymamarin/kaggriculture-episodes`](https://www.kaggle.com/datasets/georgymamarin/kaggriculture-episodes) for the field, Apache-2.0, and [`destbreso/kaggriculture-top10-replay-streams`](https://www.kaggle.com/datasets/destbreso/kaggriculture-top10-replay-streams) for the within-agent control, CC0.

**Every count here is a lower bound and every birth date an upper bound**, because the dataset is a lagging sample of the ladder and its coverage by submission is uneven.

**Reference.** Holland, Huber, Dress and Moulton, *delta Plots: A Tool for Analyzing Phylogenetic Distance Data*, Molecular Biology and Evolution 19(12), 2002.